In [ ]:
# 2025AC05115.ipynb

import numpy as np

# Step 1: Metrics
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def precision(y_true, y_pred):
    tp = np.sum((y_true==1) & (y_pred==1))
    fp = np.sum((y_true==0) & (y_pred==1))
    return tp / (tp+fp+1e-9)

def recall(y_true, y_pred):
    tp = np.sum((y_true==1) & (y_pred==1))
    fn = np.sum((y_true==1) & (y_pred==0))
    return tp / (tp+fn+1e-9)

def f1_score(y_true, y_pred):
    p, r = precision(y_true, y_pred), recall(y_true, y_pred)
    return 2*p*r / (p+r+1e-9)

def mcc(y_true, y_pred):
    tp = np.sum((y_true==1) & (y_pred==1))
    tn = np.sum((y_true==0) & (y_pred==0))
    fp = np.sum((y_true==0) & (y_pred==1))
    fn = np.sum((y_true==1) & (y_pred==0))
    num = (tp*tn - fp*fn)
    den = np.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn))
    return num / (den+1e-9)

def auc_score(y_true, y_prob):
    thresholds = np.sort(y_prob)
    tpr, fpr = [], []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        tp = np.sum((y_true==1) & (y_pred==1))
        fp = np.sum((y_true==0) & (y_pred==1))
        fn = np.sum((y_true==1) & (y_pred==0))
        tn = np.sum((y_true==0) & (y_pred==0))
        tpr.append(tp/(tp+fn+1e-9))
        fpr.append(fp/(fp+tn+1e-9))
    return np.trapz(tpr, fpr)

# Step 2: Models

# Logistic Regression
class LogisticRegressionScratch:
    def __init__(self, lr=0.01, n_iter=5000):
        self.lr = lr
        self.n_iter = n_iter
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
    
    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        self.b = 0
        for _ in range(self.n_iter):
            linear = np.dot(X, self.w) + self.b
            y_pred = self.sigmoid(linear)
            dw = np.dot(X.T, (y_pred - y)) / len(y)
            db = np.sum(y_pred - y) / len(y)
            self.w -= self.lr * dw
            self.b -= self.lr * db
    
    def predict_proba(self, X):
        return self.sigmoid(np.dot(X, self.w) + self.b)
    
    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

# KNN
class KNN:
    def __init__(self, k=5):
        self.k = k
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
    
    def predict(self, X):
        preds = []
        for x in X:
            distances = np.linalg.norm(self.X_train - x, axis=1)
            idx = np.argsort(distances)[:self.k]
            votes = self.y_train[idx]
            preds.append(np.round(np.mean(votes)))
        return np.array(preds)

# Naive Bayes (Gaussian)
class GaussianNB:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.mean = {}
        self.var = {}
        self.prior = {}
        for c in self.classes:
            X_c = X[y==c]
            self.mean[c] = X_c.mean(axis=0)
            self.var[c] = X_c.var(axis=0) + 1e-9
            self.prior[c] = len(X_c) / len(X)
    
    def predict(self, X):
        preds = []
        for x in X:
            posteriors = []
            for c in self.classes:
                prior = np.log(self.prior[c])
                likelihood = -0.5*np.sum(np.log(2*np.pi*self.var[c]))
                likelihood -= 0.5*np.sum(((x-self.mean[c])**2)/self.var[c])
                posteriors.append(prior+likelihood)
            preds.append(np.argmax(posteriors))
        return np.array(preds)

# Decision Tree
class DecisionTree:
    def __init__(self, max_depth=3):
        self.max_depth = max_depth
    
    def gini(self, y):
        classes, counts = np.unique(y, return_counts=True)
        return 1 - np.sum((counts/len(y))**2)
    
    def split(self, X, y, feature, threshold):
        left_idx = X[:,feature] <= threshold
        right_idx = X[:,feature] > threshold
        return X[left_idx], y[left_idx], X[right_idx], y[right_idx]
    
    def best_split(self, X, y):
        best_feat, best_thresh, best_score = None, None, 1e9
        for feat in range(X.shape[1]):
            thresholds = np.unique(X[:,feat])
            for t in thresholds:
                X_left, y_left, X_right, y_right = self.split(X,y,feat,t)
                if len(y_left)==0 or len(y_right)==0: continue
                score = (len(y_left)*self.gini(y_left) + len(y_right)*self.gini(y_right)) / len(y)
                if score < best_score:
                    best_feat, best_thresh, best_score = feat, t, score
        return best_feat, best_thresh
    
    def build(self, X, y, depth):
        if depth==self.max_depth or len(np.unique(y))==1:
            return int(np.argmax(np.bincount(y)))
        feat, thresh = self.best_split(X,y)
        if feat is None: return int(np.argmax(np.bincount(y)))
        X_left,y_left,X_right,y_right = self.split(X,y,feat,thresh)
        return {"feat":feat,"thresh":thresh,
                "left":self.build(X_left,y_left,depth+1),
                "right":self.build(X_right,y_right,depth+1)}
    
    def fit(self,X,y):
        self.tree = self.build(X,y,0)
    
    def predict_one(self,x,node):
        if isinstance(node,int): return node
        if x[node["feat"]] <= node["thresh"]:
            return self.predict_one(x,node["left"])
        else:
            return self.predict_one(x,node["right"])
    
    def predict(self,X):
        return np.array([self.predict_one(x,self.tree) for x in X])

# Random Forest
class RandomForest:
    def __init__(self,n_estimators=10,max_depth=3):
        self.n_estimators=n_estimators
        self.max_depth=max_depth
    
    def fit(self,X,y):
        self.trees=[]
        n=len(X)
        for _ in range(self.n_estimators):
            idx=np.random.choice(n,n,replace=True)
            X_s,y_s=X[idx],y[idx]
            tree=DecisionTree(max_depth=self.max_depth)
            tree.fit(X_s,y_s)
            self.trees.append(tree)
    
    def predict(self,X):
        preds=np.array([tree.predict(X) for tree in self.trees])
        return np.round(np.mean(preds,axis=0)).astype(int)

                 Model  Accuracy       AUC  Precision    Recall  F1 Score  \
0  logistic_regression  0.733333  0.824327   0.754941  0.743191  0.749020   
1        decision_tree  0.779167  0.777163   0.787072  0.805447  0.796154   
2                  knn  0.731250  0.797247   0.744275  0.758755  0.751445   
3          naive_bayes  0.722917  0.797979   0.767241  0.692607  0.728016   
4        random_forest  0.797917  0.877676   0.807692  0.817121  0.812379   

        MCC  
0  0.464678  
1  0.555491  
2  0.459088  
3  0.449573  
4  0.593480  
